# 09_track3_triage: Objective 3, auto-tagging and reassignment risk
Objective 3 asks to auto-tag tickets with the right priority and department, so that reassignment and its delay are reduced. The extract has no department, assignment-group or resolver field, so department tagging cannot be trained (requested from the data owner on 4 Nov 2026). This notebook covers the two parts the data supports, both scored when a ticket is created:
- Sections 1 to 4: reassignment risk (will a ticket be reassigned at least once?).
- Section 5: priority auto-tagging (P2 to P5).
- Section 6: findings and decision.

Aggregates only: WBS and subcategory codes are not printed. Needs `data/processed/` (itsm_clean.pkl, track1_train.pkl, track1_test.pkl) and xgboost; sections 3 and 5 take a few minutes.

## 1. Setup and the reassignment target

In [1]:
import sys
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

sys.path.append("../src")
from features import set_b
from project_config import CUT, START, TEST_START

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# Reassignment target: opened inside the modelling window, reassignment count recorded
df = pd.read_pickle("../data/processed/itsm_clean.pkl")
w = df[(df["Open_Time"] >= START) & (df["Open_Time"] < CUT) & df["No_of_Reassignments"].notna()].copy()
r = w["No_of_Reassignments"].astype(int)
w["reassigned"] = (r >= 1).astype(int)
w["reassigned_2plus"] = (r >= 2).astype(int)
print("Tickets:", len(w), "| reassigned at least once:", int(w["reassigned"].sum()))

Tickets: 42303 | reassigned at least once: 17202


<details>
<summary><b>Code notes, line by line</b></summary>

- `import ... spearmanr ... sklearn ... XGBClassifier` : Rank correlation, model building, metrics and the five algorithms.
- `from features import set_b; from project_config import CUT, START, TEST_START` : Shared feature set and dates.
- `w = df[(Open_Time >= START) & (Open_Time < CUT) & No_of_Reassignments.notna()]` : Window tickets with a recorded reassignment count.
- `r = w["No_of_Reassignments"].astype(int)` : Count as whole numbers.
- `w["reassigned"] = (r >= 1); w["reassigned_2plus"] = (r >= 2)` : Target (at least one hand-off) and a second, stricter flag for the profile.

</details>

### 1b. Is there a department or routing field?
The column names are searched for routing terms.

In [2]:
terms = ["department", "dept", "assign", "group", "team", "owner", "resolver", "queue"]
raw_cols = pd.read_csv("../data/raw/itsm_raw.csv", nrows=0).columns
print("Columns in the extract:", len(raw_cols), "| names containing a routing term:",
      [c for c in raw_cols if any(t in c.lower() for t in terms)])

Columns in the extract: 25 | names containing a routing term: ['No_of_Reassignments']


<details>
<summary><b>Code notes, line by line</b></summary>

- `terms = ["department", "dept", "assign", ...]` : Words that would indicate a routing field.
- `raw_cols = pd.read_csv(..., nrows=0).columns` : Column names of the raw extract (header only).
- `[c for c in raw_cols if any(t in c.lower() for t in terms)]` : Columns whose name contains any of the words.

</details>

**Result.** The only match is No_of_Reassignments, a count of hand-offs, not the team that handled the ticket. No department, assignment-group or resolver field exists, so department tagging cannot be trained.

## 2. Reassignment profile

In [3]:
# Table 1: overall shares
t1 = pd.DataFrame({"Value": [f"{len(w):,}",
                             f"{w['reassigned'].mean() * 100:.1f}%",
                             f"{w['reassigned_2plus'].mean() * 100:.1f}%"]},
                  index=["Tickets in the modelling window",
                         "Reassigned at least once (%)",
                         "Reassigned two or more times (%)"])
t1.index.name = "Measure"

# Table 2: tickets and delay by number of reassignments
order = ["0", "1", "2", "3", "4", "5+"]
t2 = (w.assign(n=r.clip(upper=5).map(lambda x: "5+" if x == 5 else str(x)))
        .groupby("n").agg(Tickets=("reassigned", "size"),
                          Median_elapsed=("Elapsed_hrs", "median"))
        .reindex(order))
t2.insert(1, "Share of tickets (%)", (t2["Tickets"] / t2["Tickets"].sum() * 100).round(1))
t2["Median_elapsed"] = t2["Median_elapsed"].round(1)
t2 = t2.rename(columns={"Median_elapsed": "Median hours open to close"})
t2.index.name = "Number of reassignments"

def rate_table(key, label, sort_by_size=True, top=None):
    g = w.groupby(key).agg(**{"Tickets": ("reassigned", "size"),
                              "Reassigned at least once (%)": ("reassigned", "mean"),
                              "Reassigned two or more times (%)": ("reassigned_2plus", "mean")})
    g.iloc[:, 1:] = (g.iloc[:, 1:] * 100).round(1)
    if sort_by_size:
        g = g.sort_values("Tickets", ascending=False)
    g.index.name = label
    return g.head(top) if top else g

t3 = rate_table(w["Open_Time"].dt.to_period("M"), "Month opened", sort_by_size=False)
t4 = rate_table("CI_Cat", "Asset type (CI_Cat)", top=6)
t5 = rate_table("Category", "Ticket category")

for title, table in [("Table 1. Overall reassignment rate", t1),
                     ("Table 2. Tickets and delay by number of reassignments", t2),
                     ("Table 3. Reassignment rate by month", t3),
                     ("Table 4. Reassignment rate by asset type (6 largest)", t4),
                     ("Table 5. Reassignment rate by ticket category", t5)]:
    print(title)
    display(table)

Table 1. Overall reassignment rate


,Value
Measure,
Tickets in the modelling window,"42,303"
Reassigned at least once (%),40.7%
Reassigned two or more times (%),25.0%


Table 2. Tickets and delay by number of reassignments


,Tickets,Share of tickets (%),Median hours open to close
Number of reassignments,,,
0,25101,59.3,2.5
1,6606,15.6,24.4
2,4856,11.5,53.6
3,2024,4.8,116.5
4,1411,3.3,140.5
5+,2305,5.4,243.6


Table 3. Reassignment rate by month


,Tickets,Reassigned at least once (%),Reassigned two or more times (%)
Month opened,,,
2013-10,8606,41.3,25.1
2013-11,8128,41.9,25.5
2013-12,6742,41.6,26.6
2014-01,8270,40.3,25.0
2014-02,7377,38.7,24.0
2014-03,3180,39.2,22.9


Table 4. Reassignment rate by asset type (6 largest)


,Tickets,Reassigned at least once (%),Reassigned two or more times (%)
Asset type (CI_Cat),,,
application,29797,43.0,27.5
subapplication,7060,29.3,16.2
computer,3377,50.9,27.6
storage,635,16.4,7.7
hardware,399,38.6,20.6
software,303,30.7,17.2


Table 5. Reassignment rate by ticket category


,Tickets,Reassigned at least once (%),Reassigned two or more times (%)
Ticket category,,,
incident,34134,38.6,22.0
request for information,8159,49.4,37.8
complaint,10,10.0,10.0


<details>
<summary><b>Code notes, line by line</b></summary>

- `t1 = pd.DataFrame({"Value": [...]}, index=[...])` : Table 1: tickets, % reassigned at least once, % reassigned twice or more.
- `r.clip(upper=5).map(lambda x: "5+" if x == 5 else str(x))` : Groups 5 or more hand-offs into '5+'.
- `groupby("n").agg(Tickets=..., Median_elapsed=("Elapsed_hrs", "median"))` : Table 2: tickets and median hours to close per number of hand-offs.
- `t2.insert(1, "Share of tickets (%)", ...)` : Adds each group's share.
- `def rate_table(key, label, sort_by_size=True, top=None):` : Reusable table of reassignment rates by any grouping.
- `t3 = rate_table(month...); t4 = rate_table("CI_Cat", top=6); t5 = rate_table("Category")` : Tables 3 to 5: by month, by asset type, by category.
- `for title, table in [...]: print(title); display(table)` : Shows the five tables with numbered titles.

</details>

**Result.** 40.7% of window tickets are reassigned at least once and 25.0% twice or more (Table 1). Median time to close rises with every hand-off: 2.5 hours with none, 24.4 with one, 53.6 with two and 243.6 with five or more (Table 2). The rate is steady by month, 38.7% to 41.9% (Table 3). Computer tickets are reassigned most (50.9%) and storage least (16.4%) (Table 4), and requests for information more than incidents (49.4% against 38.6%; Table 5).

In [4]:
rho = spearmanr(w["No_of_Reassignments"], w["Elapsed_hrs"])[0]
print(f"Spearman rank correlation, reassignments against hours open to close: {rho:.2f} ({len(w):,} tickets)")

Spearman rank correlation, reassignments against hours open to close: 0.53 (42,303 tickets)


<details>
<summary><b>Code notes, line by line</b></summary>

- `spearmanr(w["No_of_Reassignments"], w["Elapsed_hrs"])[0]` : Rank correlation between hand-offs and hours to close (robust to the long tail); [0] is the coefficient.

</details>

**Result.** Spearman rank correlation 0.53 between the number of reassignments and the hours from open to close (42,303 tickets): more hand-offs go with longer tickets. An association, not a cause: hard tickets may simply be passed on more often.

## 3. Five models with time-ordered CV, and a lookup baseline

In [5]:
TARGET = "reassigned"
w["hour"] = w["Open_Time"].dt.hour
w["weekday"] = w["Open_Time"].dt.day_of_week
w = w.sort_values("Open_Time").reset_index(drop=True)
tr, te = w[w["Open_Time"] < TEST_START], w[w["Open_Time"] >= TEST_START]

models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000),
        {"clf__C": [0.01, 0.1, 1, 10]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [3, 5, 8], "clf__min_samples_leaf": [20, 50]}),
    "Random Forest": (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        {"clf__max_depth": [6, 10], "clf__min_samples_leaf": [5, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
        {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=42, n_jobs=-1),
        {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1], "clf__min_child_weight": [1, 5]}),
}

def make_pipe(cols, clf):
    enc = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=30)
    return Pipeline([("pre", ColumnTransformer([("cat", enc, cols)])), ("clf", clf)])

def tune(train):
    X, y = train[set_b].astype(str), train[TARGET]
    wts = compute_sample_weight("balanced", y)
    fitted = {}
    for name, (clf, grid) in models.items():
        gs = GridSearchCV(make_pipe(set_b, clf), grid, cv=TimeSeriesSplit(n_splits=3),
                          scoring="average_precision", n_jobs=1)
        gs.fit(X, y, clf__sample_weight=wts)
        fitted[name] = gs
    return fitted

def evaluate(fitted, train, test):
    Xtr, Xte = train[set_b].astype(str), test[set_b].astype(str)
    ytr, yte = train[TARGET], test[TARGET]
    rows = []
    for name, gs in fitted.items():
        m = gs.best_estimator_
        pred_tr, pred_te = m.predict(Xtr), m.predict(Xte)
        proba = m.predict_proba(Xte)[:, 1]
        rows.append({"Model": name,
                     "CV AP (train)": gs.best_score_,
                     "Train accuracy": accuracy_score(ytr, pred_tr),
                     "Test accuracy": accuracy_score(yte, pred_te),
                     "Recall": recall_score(yte, pred_te),
                     "Precision": precision_score(yte, pred_te, zero_division=0),
                     "F1": f1_score(yte, pred_te),
                     "ROC-AUC": roc_auc_score(yte, proba),
                     "Avg precision": average_precision_score(yte, proba)})
    best = max(fitted, key=lambda k: fitted[k].best_score_)
    report = classification_report(yte, fitted[best].best_estimator_.predict(Xte),
                                   target_names=["Not reassigned", "Reassigned"], digits=3, zero_division=0)
    return pd.DataFrame(rows).set_index("Model").round(3), best, report

fitted_t3 = tune(tr)
res_t3, best_t3, report_t3 = evaluate(fitted_t3, tr, te)

# Simple non-ML baseline: each test ticket gets the training reassignment rate of its CI_Cat x Category
lookup = tr.groupby(["CI_Cat", "Category"])[TARGET].mean().rename("lookup")
lk = (te[["CI_Cat", "Category"]].merge(lookup, left_on=["CI_Cat", "Category"], right_index=True, how="left")
        ["lookup"].fillna(tr[TARGET].mean()))
baseline = pd.DataFrame({"ROC-AUC": [roc_auc_score(te[TARGET], lk)],
                         "Avg precision": [average_precision_score(te[TARGET], lk)]},
                        index=["Lookup: rate by CI_Cat x Category"]).round(3)

print(f"Train {len(tr):,} tickets ({tr[TARGET].mean() * 100:.1f}% reassigned) | "
      f"Test {len(te):,} tickets ({te[TARGET].mean() * 100:.1f}% reassigned) | "
      f"always-'not reassigned' accuracy {1 - te[TARGET].mean():.3f}")
print("Table 6. Five algorithms, tuned with time-ordered CV, scored on the test period")
display(res_t3)
print("Table 7. Simple lookup baseline (no machine learning)")
display(baseline)
print(f"Classification report, best model by CV average precision: {best_t3}")
print(report_t3)
print("Table 8. Best grid-search settings")
display(pd.Series({k: str(g.best_params_) for k, g in fitted_t3.items()}, name="Best settings").to_frame())

Train 31,746 tickets (41.3% reassigned) | Test 10,557 tickets (38.9% reassigned) | always-'not reassigned' accuracy 0.611
Table 6. Five algorithms, tuned with time-ordered CV, scored on the test period


,CV AP (train),Train accuracy,Test accuracy,Recall,Precision,F1,ROC-AUC,Avg precision
Model,,,,,,,,
Logistic Regression,0.630,0.678,0.701,0.584,0.624,0.603,0.728,0.637
Decision Tree,0.560,0.659,0.693,0.336,0.728,0.460,0.655,0.570
Random Forest,0.603,0.679,0.695,0.461,0.653,0.540,0.712,0.649
Gradient Boosting,0.635,0.688,0.700,0.476,0.658,0.553,0.726,0.653
XGBoost,0.641,0.685,0.701,0.497,0.651,0.564,0.728,0.659


Table 7. Simple lookup baseline (no machine learning)


,ROC-AUC,Avg precision
Lookup: rate by CI_Cat x Category,0.606,0.456


Classification report, best model by CV average precision: XGBoost
                precision    recall  f1-score   support

Not reassigned      0.722     0.831     0.773      6454
    Reassigned      0.651     0.497     0.564      4103

      accuracy                          0.701     10557
     macro avg      0.687     0.664     0.668     10557
  weighted avg      0.695     0.701     0.692     10557

Table 8. Best grid-search settings


,Best settings
Logistic Regression,{'clf__C': 10}
Decision Tree,"{'clf__max_depth': 8, 'clf__min_samples_leaf':..."
Random Forest,"{'clf__max_depth': 10, 'clf__min_samples_leaf'..."
Gradient Boosting,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."
XGBoost,"{'clf__learning_rate': 0.1, 'clf__max_depth': ..."


<details>
<summary><b>Code notes, line by line</b></summary>

- `TARGET = "reassigned"` : The column predicted.
- `w["hour"], w["weekday"]; w.sort_values("Open_Time")` : Time features and time order for TimeSeriesSplit.
- `tr, te = w[< TEST_START], w[>= TEST_START]` : Same split dates as Track 1.
- `models = {...}` : Five algorithms with small grids (as in 06).
- `def make_pipe(cols, clf):` : One-hot encoder (rare values pooled) + model.
- `def tune(train):` : Grid search per algorithm: 3 time-ordered folds, scored by average precision, balanced weights.
- `def evaluate(fitted, train, test):` : Train/test accuracy, recall, precision, F1 at 0.5, ROC-AUC and AP per model; report for the CV-best model.
- `lookup = tr.groupby(["CI_Cat", "Category"])[TARGET].mean()` : Non-ML baseline: each combination's training reassignment rate.
- `te[...].merge(lookup, ..., how="left")["lookup"].fillna(tr[TARGET].mean())` : Gives each test ticket its combination's rate; unseen combinations get the overall rate.
- `baseline = pd.DataFrame({"ROC-AUC": ..., "Avg precision": ...})` : Table 7: the baseline's scores.
- `display(res_t3); display(baseline); print(report_t3)` : Tables 6 and 7 and the classification report.
- `pd.Series({k: str(g.best_params_) ...})` : Table 8: best settings per algorithm.

</details>

**Result.** On the test period all five tuned models beat the lookup baseline (Table 7: ROC-AUC 0.606, average precision 0.456): ROC-AUC 0.655 to 0.728 and average precision 0.570 to 0.659 (Table 6). Logistic regression (0.728, 0.637) and XGBoost (0.728, 0.659) tie on ROC-AUC; XGBoost is about 0.01 higher in CV average precision (0.641 against 0.630). Test accuracy (0.693 to 0.701) is only modestly above always predicting "not reassigned" (0.611), so the models are used as a ranking (section 4). The classification report shows XGBoost at the 0.5 cut-off finds about half of the reassigned tickets (recall 0.497) at precision 0.651; Table 8 lists the chosen settings. Logistic regression is used in section 4 because its coefficients show what drives the score.

## 4. Operating points and what drives the score

In [6]:
chosen = "Logistic Regression"
m_t3 = fitted_t3[chosen].best_estimator_
p_t3 = m_t3.predict_proba(te[set_b].astype(str))[:, 1]
y_t3 = te[TARGET].to_numpy()
base = y_t3.mean()

rows = []
for share in [0.10, 0.20, 0.30, 0.40]:
    k = int(round(share * len(p_t3)))
    caught = int(y_t3[np.argsort(-p_t3)[:k]].sum())
    rows.append({"Tickets flagged (%)": int(share * 100),
                 "Tickets flagged": k,
                 "Reassigned among flagged (%)": round(caught / k * 100, 1),
                 "Lift over base rate": round(caught / k / base, 2),
                 "Share of all reassigned tickets caught (%)": round(caught / y_t3.sum() * 100, 1)})
print(f"Table 9. {chosen}: flagging the highest-risk tickets (test base rate {base * 100:.1f}%)")
display(pd.DataFrame(rows).set_index("Tickets flagged (%)"))

pre, clf = m_t3.named_steps["pre"], m_t3.named_steps["clf"]
coef = pd.Series(clf.coef_[0], index=pre.get_feature_names_out())
groups = pd.DataFrame([{"Input column": c,
                        "One-hot features": int(coef.index.str.startswith(f"cat__{c}_").sum()),
                        "Mean |coef|": round(coef[coef.index.str.startswith(f"cat__{c}_")].abs().mean(), 3)}
                       for c in set_b]).set_index("Input column")
print("Table 10. Weight by input column (rough guide)")
display(groups)

show = coef[coef.index.str.contains(r"cat__(?:CI_Cat|Category|hour|weekday)_")]
pos, neg = show.sort_values(ascending=False).head(8).round(2), show.sort_values().head(8).round(2)
label = lambda s: s.index.str.replace("cat__", "", regex=False).str.replace("_infrequent_sklearn", " (rare values pooled)", regex=False)
drivers = pd.DataFrame({"Towards reassignment": label(pos), "Coef (towards)": pos.values,
                        "Away from reassignment": label(neg), "Coef (away)": neg.values},
                       index=range(1, 9))
print("Table 11. Strongest asset type, category and time effects (WBS and subcategory codes not shown)")
display(drivers)

Table 9. Logistic Regression: flagging the highest-risk tickets (test base rate 38.9%)


,Tickets flagged,Reassigned among flagged (%),Lift over base rate,Share of all reassigned tickets caught (%)
Tickets flagged (%),,,,
10,1056,78.3,2.02,20.2
20,2111,71.7,1.85,36.9
30,3167,65.2,1.68,50.4
40,4223,59.8,1.54,61.6


Table 10. Weight by input column (rough guide)


,One-hot features,Mean |coef|
Input column,,
CI_Cat,12,0.387
CI_Subcat,27,0.522
WBS,101,0.971
Category,3,0.110
hour,18,0.263
weekday,7,0.090


Table 11. Strongest asset type, category and time effects (WBS and subcategory codes not shown)


,Towards reassignment,Coef (towards),Away from reassignment,Coef (away)
1,CI_Cat (rare values pooled),1.40,CI_Cat_networkcomponents,-1.28
2,hour_21,1.14,CI_Cat_database,-0.75
3,hour_0,0.66,hour_19,-0.54
4,hour_1,0.38,hour_6,-0.32
5,CI_Cat_displaydevice,0.26,weekday_5,-0.30
6,CI_Cat_computer,0.22,hour_7,-0.27
7,CI_Cat_application,0.22,CI_Cat_Unknown,-0.19
8,Category_incident,0.14,hour_11,-0.18


<details>
<summary><b>Code notes, line by line</b></summary>

- `chosen = "Logistic Regression"` : Used for the ranking because its coefficients can be read.
- `p_t3 = m_t3.predict_proba(...)[:, 1]; base = y_t3.mean()` : Test risk scores and the base rate.
- `for share in [0.10, 0.20, 0.30, 0.40]:` : Flag the highest-risk share of tickets.
- `caught = int(y_t3[np.argsort(-p_t3)[:k]].sum())` : Reassigned tickets among the flagged ones.
- `"Lift over base rate": caught / k / base` : How many times the base rate the flagged group reaches.
- `coef = pd.Series(clf.coef_[0], index=pre.get_feature_names_out())` : Coefficients by one-hot column.
- `groups = ... "Mean |coef|"` : Table 10: average weight per input column (mean, so columns with many codes are not inflated).
- `show = coef[...CI_Cat|Category|hour|weekday...]` : Readable columns only (codes not printed).
- `label = lambda s: ...replace("_infrequent_sklearn", " (rare values pooled)")` : Friendlier names in Table 11.
- `drivers = pd.DataFrame({...})` : Table 11: eight strongest effects in each direction.

</details>

**Result.** Flagging the 10% highest-risk test tickets finds 78.3% reassigned (2.02 times the 38.9% base rate) and catches 20.2% of all reassigned tickets; the top 20% gives 71.7% (1.85 times) and catches 36.9% (Table 9). WBS carries the largest weights (mean |coef| 0.97, Table 10). Towards reassignment: rare asset types, late-evening and night hours (21:00 to 01:00); away from it: network components and database tickets (Table 11). Associations, not causes.

## 5. Priority auto-tagging (P2 to P5)
A four-class model that suggests a priority when a ticket is created, from the same asset and time fields as Track 1 (Set B, no Impact or Urgency). Same data and split as Track 1 (`05_split_features`): train October to January, test 1 February to 16 March. P1 is left out: its 3 tickets all fall in the test period, so it cannot be learned. Models are tuned by macro-F1 (every class counts equally) with time-ordered CV and balanced weights; two baselines show what simple rules achieve. Model choice uses Track 1's rule: the best CV macro-F1, but logistic regression is kept if it is within 0.01.

In [7]:
train_t = pd.read_pickle("../data/processed/track1_train.pkl")
test_t = pd.read_pickle("../data/processed/track1_test.pkl")
PRIORITIES = [2, 3, 4, 5]
tr_p = train_t[train_t["Priority"].isin(PRIORITIES)]
te_p = test_t[test_t["Priority"].isin(PRIORITIES)]
p1_te = test_t[test_t["Priority"] == 1]
Xtr_p, Xte_p = tr_p[set_b].astype(str), te_p[set_b].astype(str)
ytr_p = tr_p["Priority"].astype(int) - 2                  # P2..P5 -> 0..3
yte_p = (te_p["Priority"].astype(int) - 2).to_numpy()
names = [f"P{p}" for p in PRIORITIES]
print("Table 12. Tickets per priority")
display(pd.DataFrame({"Train": tr_p["Priority"].astype(int).value_counts().sort_index(),
                      "Test": te_p["Priority"].astype(int).value_counts().sort_index()}).rename(index=lambda p: f"P{p}"))

tag_models = {
    "Logistic Regression": (LogisticRegression(max_iter=3000), {"clf__C": [0.01, 0.1, 1, 10]}),
    "Random Forest": (RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
        {"clf__max_depth": [10, 20], "clf__min_samples_leaf": [5, 20]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=42),
        {"clf__learning_rate": [0.05, 0.1], "clf__max_depth": [2, 3]}),
    "XGBoost": (XGBClassifier(objective="multi:softprob", n_estimators=200, eval_metric="mlogloss",
                              random_state=42, n_jobs=-1),
        {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]}),
}
wts_p = compute_sample_weight("balanced", ytr_p)
fitted_tag = {}
for name, (clf, grid) in tag_models.items():
    gs = GridSearchCV(make_pipe(set_b, clf), grid, cv=TimeSeriesSplit(n_splits=3), scoring="f1_macro", n_jobs=1)
    gs.fit(Xtr_p, ytr_p, clf__sample_weight=wts_p)
    fitted_tag[name] = gs

def tag_row(name, pred, cv=np.nan):
    return {"Model": name, "CV macro-F1 (train)": cv,
            "Test accuracy": accuracy_score(yte_p, pred),
            "Balanced accuracy": balanced_accuracy_score(yte_p, pred),
            "Macro-F1": f1_score(yte_p, pred, average="macro"),
            "Weighted F1": f1_score(yte_p, pred, average="weighted"),
            "P2 recall": recall_score(yte_p, pred, labels=[0], average="macro", zero_division=0)}

# Baselines: always the most common priority; the most common priority of the ticket's CI_Subcat x Category
majority = int(ytr_p.mode().iloc[0])
lookup_p = (tr_p.assign(y=ytr_p).groupby(["CI_Subcat", "Category"])["y"]
                .agg(lambda s: s.mode().iloc[0]).rename("lookup"))
pred_lookup = (te_p[["CI_Subcat", "Category"]]
               .merge(lookup_p, left_on=["CI_Subcat", "Category"], right_index=True, how="left")["lookup"]
               .fillna(majority).astype(int).to_numpy())

rows = [tag_row(n, g_.best_estimator_.predict(Xte_p), g_.best_score_) for n, g_ in fitted_tag.items()]
rows += [tag_row("Baseline: always the most common priority", np.full(len(yte_p), majority)),
         tag_row("Baseline: most common priority per CI_Subcat x Category", pred_lookup)]
print(f"Table 13. Priority tagging, test period ({len(yte_p):,} tickets)")
display(pd.DataFrame(rows).set_index("Model").round(3))

cv_f1 = {k: g_.best_score_ for k, g_ in fitted_tag.items()}
top = max(cv_f1, key=cv_f1.get)
tag_name = "Logistic Regression" if cv_f1[top] - cv_f1["Logistic Regression"] < 0.01 else top
tagger = fitted_tag[tag_name].best_estimator_
pred_tag = tagger.predict(Xte_p)
print(f"Top by CV macro-F1: {top} ({cv_f1[top]:.3f}) | chosen: {tag_name} ({cv_f1[tag_name]:.3f})")
print(classification_report(yte_p, pred_tag, labels=[0, 1, 2, 3], target_names=names, digits=3, zero_division=0))
print("Table 14. Actual against suggested priority, chosen model")
display(pd.DataFrame(confusion_matrix(yte_p, pred_tag, labels=[0, 1, 2, 3]),
                     index=[f"Actual {n}" for n in names], columns=[f"Suggested {n}" for n in names]))
print("The", len(p1_te), "P1 test tickets were suggested as:",
      pd.Series(tagger.predict(p1_te[set_b].astype(str)) + 2).map(lambda p: f"P{p}").value_counts().to_dict())

Table 12. Tickets per priority


,Train,Test
Priority,,
P2,541,112
P3,3517,1250
P4,15180,5347
P5,11607,3517


Table 13. Priority tagging, test period (10,226 tickets)


,CV macro-F1 (train),Test accuracy,Balanced accuracy,Macro-F1,Weighted F1,P2 recall
Model,,,,,,
Logistic Regression,0.619,0.761,0.736,0.617,0.779,0.661
Random Forest,0.628,0.754,0.725,0.625,0.770,0.607
Gradient Boosting,0.629,0.762,0.736,0.628,0.778,0.643
XGBoost,0.631,0.767,0.740,0.628,0.784,0.643
Baseline: always the most common priority,NaN,0.523,0.250,0.172,0.359,0.000
Baseline: most common priority per CI_Subcat x Category,NaN,0.721,0.546,0.545,0.687,0.527


Top by CV macro-F1: XGBoost (0.631) | chosen: XGBoost (0.631)
              precision    recall  f1-score   support

          P2      0.184     0.643     0.286       112
          P3      0.473     0.790     0.592      1250
          P4      0.859     0.777     0.816      5347
          P5      0.904     0.750     0.820      3517

    accuracy                          0.767     10226
   macro avg      0.605     0.740     0.628     10226
weighted avg      0.820     0.767     0.784     10226

Table 14. Actual against suggested priority, chosen model


,Suggested P2,Suggested P3,Suggested P4,Suggested P5
Actual P2,72,23,11,6
Actual P3,55,987,139,69
Actual P4,211,780,4152,204
Actual P5,53,295,533,2636


The 3 P1 test tickets were suggested as: {'P2': 2, 'P3': 1}


<details>
<summary><b>Code notes, line by line</b></summary>

- `train_t / test_t = pd.read_pickle(track1_...)` : Same labelled data and split as Track 1.
- `PRIORITIES = [2, 3, 4, 5]; tr_p / te_p` : P1 left out (no P1 in training).
- `p1_te = test_t[test_t["Priority"] == 1]` : The P1 test tickets, checked at the end.
- `ytr_p = Priority - 2` : Classes 0 to 3 (XGBoost needs labels starting at 0).
- `display(pd.DataFrame({"Train": ..., "Test": ...}))` : Table 12: tickets per priority.
- `tag_models = {...}` : Four multi-class models; XGBoost uses multi:softprob (a probability per class).
- `wts_p = compute_sample_weight("balanced", ytr_p)` : Equal total weight per priority class.
- `GridSearchCV(..., scoring="f1_macro")` : Tuned by macro-F1, the average F1 over the four classes, so the small P2 class counts as much as P4.
- `def tag_row(name, pred, cv=np.nan):` : Accuracy, balanced accuracy (mean recall per class), macro-F1, weighted F1 and P2 recall.
- `majority = int(ytr_p.mode().iloc[0])` : Baseline 1: always the most common priority.
- `lookup_p = ...groupby(["CI_Subcat", "Category"])["y"].agg(mode)` : Baseline 2: the most common priority per subcategory and category.
- `display(pd.DataFrame(rows)...)` : Table 13: models and baselines.
- `tag_name = "Logistic Regression" if ... < 0.01 else top` : Track 1's tie-break rule.
- `classification_report(...); confusion_matrix(...)` : Per-class scores and Table 14 (actual against suggested).
- `tagger.predict(p1_te[set_b]...) + 2` : What priority the tagger suggests for the P1 tickets.

</details>

**Result.** All four models clearly beat both baselines (Table 13): macro-F1 0.617 to 0.628 and balanced accuracy 0.725 to 0.740, against 0.545 and 0.546 for the CI_Subcat x Category lookup and 0.172 and 0.250 for always suggesting P4. XGBoost has the best CV macro-F1 (0.631 against 0.619 for logistic regression, so the 0.01 rule selects it). On the test period it reaches accuracy 0.767, macro-F1 0.628 and balanced accuracy 0.740; gradient boosting is level on test macro-F1 (0.628), so the choice between the two tree models is within noise. P4 and P5 are tagged well (F1 0.82); P3 (F1 0.59) and P2 (recall 0.64, precision 0.18) are the hard classes, because they share asset codes with many P4 tickets, so a share of P4 tickets is suggested as P3 or P2 (Table 14).

## 6. Findings and decision (Objective 3)
1. **Department tagging is not possible with this extract.** There is no department, assignment-group or resolver field (section 1b: the only matching column is No_of_Reassignments), and clustering asset codes would only invent groups that cannot be validated. The field has been requested.
2. **Reassignment matters.** 40.7% of window tickets are reassigned at least once and 25.0% twice or more (Table 1). Median time to close rises from 2.5 hours with no reassignment to 243.6 hours with five or more (Table 2; Spearman correlation 0.53, section 2; an association, not a cause). The rate is stable by month (Table 3) and highest for computer tickets and requests for information (Tables 4 and 5).
3. **Reassignment risk can be scored at creation.** Five tuned models reach ROC-AUC 0.655 to 0.728 on the test period (Table 6), against 0.606 for a CI_Cat x Category lookup (Table 7). The top 10% of logistic-regression scores are 78.3% reassigned (2.02 times the base rate) and catch 20.2% of reassigned tickets; the top 20% catch 36.9% (Table 9). WBS, rare asset types and night hours drive the score (Tables 10 and 11).
4. **Priority can be suggested at creation (section 5).** A four-class XGBoost model beats the lookup baseline by 0.08 in macro-F1 (0.628 against 0.545) and 0.19 in balanced accuracy (0.740 against 0.546), with accuracy 0.767 (Table 13). P2 and P3 remain the weakest classes (P2 recall 0.64 at precision 0.18; Table 14 shows 211 P4 tickets suggested as P2). It complements Track 1: the Track 1 model stays the P1/P2 alert (fixed threshold, about 5 alerts a day), while the tagger proposes a priority for every ticket. P1 cannot be learned (3 tickets, none in training; Table 12).

### Decision
- **Reassignment risk:** logistic regression on Set B, used as a ranked list (top 10% or 20%, Table 9) for an early routing check. XGBoost is the first alternative to re-test with more data.
- **Priority tag:** XGBoost on Set B (Table 13).
- **Department tag:** deferred until an assignment-group field is available.